<a href="https://colab.research.google.com/github/BosenkoTM/ClickHouse/blob/main/ch_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Аналитика успеваемости школьников — рабочая тетрадь

**Назначение тетради.** Перед тем как переносить визуализации в Yandex DataLens, мы хотим *увидеть глазами*, как они будут выглядеть, и убедиться, что данные корректно загрузились в ClickHouse, а метрики и чарты считаются правильно.

**Сценарий выполнения тетради:**

1. **CSV → DataFrame.** Читаем `Student_performance_data _.csv` в `pandas.DataFrame`.
2. **DataFrame → ClickHouse.** Подключаемся к боевой БД, создаём таблицу `raw_student_performance` и заливаем данные.
3. **Проверка успешности загрузки.** Сравниваем количество строк и агрегаты в `pandas` и в ClickHouse — если они сходятся, значит данные доехали без потерь.
4. **VIEW в ClickHouse.** Запускаем `CREATE OR REPLACE VIEW v_student_performance_dash` — это та же самая витрина, на которой потом сядет DataLens.
5. **Чарты на витрине из ClickHouse.** Тянем витрину обратно в `pandas` и строим те же 5 чартов + Treemap, что будут в DataLens, чтобы заранее отрепетировать дашборд.

> ⚠️ Тетрадь требует **реальный** ClickHouse и **реальный** CSV — синтетических данных тут больше нет. Подключение настраивается в Этапе 1б; без рабочих учётных данных дальнейшие шаги не выполнятся.

## ⚙️ Установка и импорт библиотек

В Google Colab `pandas` и `plotly` уже стоят, остальное ставим явно.

In [ ]:
!pip install -q clickhouse-connect plotly pandas ipywidgets

In [ ]:
import os
import numpy as np
import pandas as pd
import clickhouse_connect

import plotly.graph_objects as go
import plotly.express as px

# Для интерактивных фильтров (имитация селекторов DataLens)
import ipywidgets as widgets
from IPython.display import display

# Удобное отображение plotly в Colab/Jupyter
import plotly.io as pio
pio.renderers.default = 'colab'  # в JupyterLab можно поменять на 'notebook'

print('Окружение готово ✔')

## 📥 Этап 1а. Загрузка CSV в `DataFrame`

Файл `Student_performance_data _.csv` должен лежать в сессионном хранилище Colab (или в текущей рабочей директории). У нас может встретиться два варианта имени — с пробелом и без — поэтому проверяем оба.

In [ ]:
POSSIBLE_PATHS = [
    'Student_performance_data _.csv',   # как в инструкции (с пробелом)
]
csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        'CSV не найден. Загрузите Student_performance_data _.csv в сессионное хранилище Colab.'
    )

df = pd.read_csv(csv_path)
print(f'✔ Прочитан CSV «{csv_path}»: {df.shape[0]} строк, {df.shape[1]} колонок')
df.head()

In [ ]:
# Краткий профиль данных перед заливкой
print('Типы данных в CSV:')
print(df.dtypes)
print()
print('Пропуски по колонкам:')
print(df.isna().sum().to_string())

### Подготовка типов под DDL

В CSV `GradeClass` приходит как `float64` (значения `0.0 … 4.0`), а в DDL ClickHouse у нас `UInt8`. Чтобы `client.insert_df()` ничего не пришлось «угадывать», явно приводим типы — сразу к тем, что объявлены в DDL.

In [ ]:
DDL_DTYPES = {
    'StudentID':         np.uint32,
    'Age':               np.uint8,
    'Gender':            np.uint8,
    'Ethnicity':         np.uint8,
    'ParentalEducation': np.uint8,
    'StudyTimeWeekly':   np.float32,
    'Absences':          np.uint8,
    'Tutoring':          np.uint8,
    'ParentalSupport':   np.uint8,
    'Extracurricular':   np.uint8,
    'Sports':            np.uint8,
    'Music':             np.uint8,
    'Volunteering':      np.uint8,
    'GPA':               np.float32,
    'GradeClass':        np.uint8,
}
df = df.astype(DDL_DTYPES)
print('После приведения:')
print(df.dtypes)

## 🔌 Этап 1б. Подключение к ClickHouse

Заполните `username`, `password` и `database` своими **выданными учётными данными** из курса. После запуска ячейка должна напечатать версию сервера — это и есть проверка живости подключения.

In [ ]:
# 🔑 ЗАМЕНИТЕ на ваши боевые учётные данные!
client = clickhouse_connect.get_client(
    host='95.131.149.21',
    port=8123,
    username='ВАШ ЛОГИН',
    password='ВАШ ПАРОЛЬ',
    database='ВАША БАЗА ДАННЫХ',
    secure=False,           # обязательно True для облачных баз
)

# ping: запрос-однострочник проверяет соединение и права на чтение
server_version = client.command('SELECT version()')
print(f'✔ Подключились к ClickHouse, версия сервера: {server_version}')

## ⬆️ Этап 1в. Создание таблицы и загрузка данных

Создаём таблицу-приёмник `raw_student_performance` и пушим в неё `DataFrame`.

> Ячейка идемпотентна: `IF NOT EXISTS` не упадёт на повторном запуске, а `TRUNCATE` гарантирует, что после повторного `insert_df` данные не задвоятся.

In [ ]:
DDL_RAW_TABLE = '''
CREATE TABLE IF NOT EXISTS raw_student_performance (
    StudentID UInt32,
    Age UInt8,
    Gender UInt8,
    Ethnicity UInt8,
    ParentalEducation UInt8,
    StudyTimeWeekly Float32,
    Absences UInt8,
    Tutoring UInt8,
    ParentalSupport UInt8,
    Extracurricular UInt8,
    Sports UInt8,
    Music UInt8,
    Volunteering UInt8,
    GPA Float32,
    GradeClass UInt8
) ENGINE = MergeTree()
ORDER BY StudentID
'''

client.command(DDL_RAW_TABLE)
client.command('TRUNCATE TABLE raw_student_performance')

client.insert_df('raw_student_performance', df)
print(f'✔ Загружено {len(df)} строк в raw_student_performance')

## ✅ Этап 1г. Проверка успешности загрузки в ClickHouse

Делаем 4 проверки — этого достаточно, чтобы быть уверенным, что данные доехали без потерь и без преобразований типа «уехало `Float32` — приехала строка».

1. **Количество строк.** В ClickHouse столько же, сколько в CSV?
2. **Схема.** Все 15 колонок на месте, типы совпадают с DDL?
3. **Сэмпл.** Глазами смотрим первые 5 строк прямо из CH.
4. **Сходимость агрегатов.** Сравниваем `AVG(GPA)`, `MIN/MAX/SUM(Absences)`, `count(distinct StudentID)` между `pandas` и `ClickHouse` — если все цифры сошлись, значит ничего не «потерялось в пути».

In [ ]:
# 1. Количество строк
ch_count = client.query('SELECT count() FROM raw_student_performance').result_rows[0][0]
print(f'CSV  : {len(df):>5}  строк')
print(f'CH   : {ch_count:>5}  строк')
assert ch_count == len(df), 'Количество строк не совпадает!'
print('✔ Количество строк сошлось')

In [ ]:
# 2. Схема таблицы в CH
schema = client.query_df('DESCRIBE TABLE raw_student_performance')
schema[['name', 'type']]

In [ ]:
# 3. Первые 5 строк — прямо из ClickHouse
client.query_df('SELECT * FROM raw_student_performance ORDER BY StudentID LIMIT 5')

In [ ]:
# 4. Сходимость агрегатов: pandas vs ClickHouse
ch_aggs = client.query_df('''
    SELECT
        count()                          AS rows,
        countDistinct(StudentID)         AS unique_students,
        round(avg(GPA), 6)               AS avg_gpa,
        round(avg(StudyTimeWeekly), 6)   AS avg_study_time,
        sum(Absences)                    AS total_absences,
        max(Absences)                    AS max_absences
    FROM raw_student_performance
''').iloc[0]

pd_aggs = pd.Series({
    'rows':            len(df),
    'unique_students': df['StudentID'].nunique(),
    'avg_gpa':         round(float(df['GPA'].mean()), 6),
    'avg_study_time':  round(float(df['StudyTimeWeekly'].mean()), 6),
    'total_absences':  int(df['Absences'].sum()),
    'max_absences':    int(df['Absences'].max()),
})

compare = pd.DataFrame({'pandas': pd_aggs, 'clickhouse': ch_aggs})
compare['match'] = compare['pandas'] == compare['clickhouse']
compare

## 🔧 Этап 2. Создание витрины `v_student_performance_dash` (VIEW)

В сырой таблице категориальные колонки лежат числами (`Gender = 0`, `GradeClass = 4`, и т. п.). На графиках в DataLens это выглядит непонятно, поэтому делаем **VIEW**, который расшифровывает коды в человеческие названия. Ту же витрину потом подцепит DataLens.

In [ ]:
SQL_VIEW = '''
CREATE OR REPLACE VIEW v_student_performance_dash AS
SELECT
    StudentID, Age, StudyTimeWeekly, Absences, GPA,

    multiIf(Gender = 0, 'Мужской',
            Gender = 1, 'Женский',
            'Неизвестно') AS Gender_Name,

    multiIf(Ethnicity = 0, 'Европеоидная',
            Ethnicity = 1, 'Афроамериканская',
            Ethnicity = 2, 'Азиатская',
            'Другая') AS Ethnicity_Name,

    multiIf(ParentalEducation = 0, 'Нет',
            ParentalEducation = 1, 'Средняя школа',
            ParentalEducation = 2, 'Неоконченное высшее',
            ParentalEducation = 3, 'Бакалавр',
            ParentalEducation = 4, 'Магистр и выше',
            'Неизвестно') AS ParentalEdu_Name,

    multiIf(ParentalSupport = 0, 'Отсутствует',
            ParentalSupport = 1, 'Низкая',
            ParentalSupport = 2, 'Средняя',
            ParentalSupport = 3, 'Высокая',
            ParentalSupport = 4, 'Очень высокая',
            'Неизвестно') AS ParentalSupport_Name,

    multiIf(GradeClass = 0, 'A (Отлично)',
            GradeClass = 1, 'B (Хорошо)',
            GradeClass = 2, 'C (Удовлетворительно)',
            GradeClass = 3, 'D (Посредственно)',
            GradeClass = 4, 'F (Неудовлетворительно)',
            'Unknown') AS Grade_Name,

    Tutoring, Extracurricular, Sports, Music, Volunteering,

    if(Tutoring = 1,        'Да', 'Нет') AS Has_Tutoring,
    if(Extracurricular = 1, 'Да', 'Нет') AS Has_Extracurricular
FROM raw_student_performance
'''

client.command(SQL_VIEW)
print('✔ VIEW v_student_performance_dash создан / обновлён')

### Проверяем VIEW

Достаём 5 первых строк — расшифрованные колонки (`Gender_Name`, `Grade_Name`, `Has_Tutoring`, …) должны быть на месте.

In [ ]:
client.query_df('''
    SELECT StudentID, Gender_Name, Grade_Name, ParentalSupport_Name,
           Has_Tutoring, Has_Extracurricular, GPA, Absences
    FROM v_student_performance_dash
    ORDER BY StudentID
    LIMIT 5
''')

### Тянем витрину в `pandas` для построения чартов

Витрина у нас компактная (~2400 строк), поэтому удобнее один раз вытянуть её в `pandas.DataFrame` и строить графики локально. В DataLens произойдёт по сути то же самое — BI пошлёт `SELECT … FROM v_student_performance_dash` в ClickHouse и нарисует чарты.

In [ ]:
v = client.query_df('SELECT * FROM v_student_performance_dash')
print(f'✔ Витрина v: {v.shape[0]} строк, {v.shape[1]} колонок')

# Зафиксированный порядок категорий — для аккуратной сортировки в чартах
GRADE_ORDER = ['A (Отлично)', 'B (Хорошо)', 'C (Удовлетворительно)',
               'D (Слабо)', 'F (Неудовлетворительно)']

v.head()

## 📐 Этап 3. Расчётные показатели для DataLens

В инструкции к Этапу 3 в DataLens мы добавляем 5 метрик. Здесь те же формулы, посчитанные в `pandas`. Это и есть числа, которые KPI-чарт в DataLens должен показать на тех же данных.

In [ ]:
def kpi_metrics(frame: pd.DataFrame) -> dict:
    """Метрики из Этапа 3 на произвольной выборке."""
    return {
        'Всего студентов':              int(frame['StudentID'].count()),
        'Средний GPA':                  round(float(frame['GPA'].mean()), 2),
        'Средний уровень прогулов':              round(float(frame['Absences'].mean()), 2),
        'Среднее время обучения':          round(float(frame['StudyTimeWeekly'].mean()), 2),
        '% занимающихся с репетитором': round(float(frame['Tutoring'].mean()) * 100, 1),
    }

kpi_metrics(v)

## 📊 Этап 4. Пять визуализаций

Каждый чарт ниже — это будущий чарт DataLens. В заголовках я повторяю описание из инструкции, чтобы можно было легко проверить соответствие.

### Чарт 1. KPI-индикаторы

| Параметр DataLens | Значение |
|-------------------|----------|
| Тип чарта | Индикатор (Indicator) |
| Показатель | `Средний GPA` |
| Дополнительный показатель | `Всего студентов` |

In [ ]:
m = kpi_metrics(v)

fig1 = go.Figure()
fig1.add_trace(go.Indicator(
    mode='number', value=m['Средний GPA'],
    number={'valueformat': '.2f', 'font': {'size': 56}},
    title={'text': 'Средний GPA', 'font': {'size': 18}},
    domain={'row': 0, 'column': 0},
))
fig1.add_trace(go.Indicator(
    mode='number', value=m['Всего студентов'],
    number={'valueformat': ',d', 'font': {'size': 56}},
    title={'text': 'Всего студентов', 'font': {'size': 18}},
    domain={'row': 0, 'column': 1},
))
fig1.update_layout(
    grid={'rows': 1, 'columns': 2},
    height=240,
    margin=dict(t=60, b=20, l=20, r=20),
    title='Чарт 1. Ключевые показатели школы',
)
fig1.show()

### Чарт 2. Влияние поддержки родителей на оценки

| Параметр DataLens | Значение |
|-------------------|----------|
| Тип чарта | Столбчатая (Bar chart) |
| Ось X | `ParentalSupport_Name` |
| Ось Y | `Средний GPA` (`AVG([GPA])`) |
| Сортировка | По `Средний GPA` (по убыванию) |

In [ ]:
g2 = (v.groupby('ParentalSupport_Name', as_index=False)['GPA']
        .mean()
        .sort_values('GPA', ascending=False))

fig2 = px.bar(
    g2,
    x='ParentalSupport_Name', y='GPA',
    text=g2['GPA'].round(2),
    title='Чарт 2. Средний GPA по уровню поддержки родителей',
    labels={'ParentalSupport_Name': 'Поддержка родителей', 'GPA': 'Средний GPA'},
)
fig2.update_traces(textposition='outside', marker_color='#5B8DEF')
fig2.update_layout(yaxis=dict(range=[0, 4]), height=420)
fig2.show()

### Чарт 3. Прогулы vs Успеваемость (точечная диаграмма)

| Параметр DataLens | Значение |
|-------------------|----------|
| Тип чарта | Scatter |
| Ось X | `Absences` (без агрегации) |
| Ось Y | `GPA` (без агрегации) |
| Цвет | `Grade_Name` |

In [ ]:
fig3 = px.scatter(
    v,
    x='Absences', y='GPA',
    color='Grade_Name',
    category_orders={'Grade_Name': GRADE_ORDER},
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Чарт 3. Прогулы vs GPA (цвет = класс успеваемости)',
    labels={'Absences': 'Количество прогулов', 'GPA': 'GPA'},
    opacity=0.65,
)
fig3.update_traces(marker=dict(size=6))
fig3.update_layout(height=520, legend_title_text='Grade_Name')
fig3.show()

### Чарт 4. Распределение классов успеваемости

| Параметр DataLens | Значение |
|-------------------|----------|
| Тип чарта | Кольцевая (Ring/Donut) |
| Цвет | `Grade_Name` |
| Показатель | `Всего студентов` (`COUNT(StudentID)`) |
| Подписи | в процентах |

In [ ]:
g4 = (v.groupby('Grade_Name', as_index=False)
        .agg(students=('StudentID', 'count')))
g4['Grade_Name'] = pd.Categorical(g4['Grade_Name'], categories=GRADE_ORDER, ordered=True)
g4 = g4.sort_values('Grade_Name')

fig4 = px.pie(
    g4, names='Grade_Name', values='students', hole=0.55,
    category_orders={'Grade_Name': GRADE_ORDER},
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Чарт 4. Распределение классов успеваемости',
)
fig4.update_traces(textinfo='percent+label', textposition='outside')
fig4.update_layout(height=480)
fig4.show()

### Чарт 5. Влияние внеучебной активности на дисциплину

| Параметр DataLens | Значение |
|-------------------|----------|
| Тип чарта | Столбчатая (или линейная) |
| Ось X | `Has_Extracurricular` |
| Ось Y | `Средние прогулы` (`AVG([Absences])`) |

In [ ]:
g5 = (v.groupby('Has_Extracurricular', as_index=False)['Absences']
        .mean()
        .sort_values('Has_Extracurricular'))

fig5 = px.bar(
    g5,
    x='Has_Extracurricular', y='Absences',
    text=g5['Absences'].round(2),
    title='Чарт 5. Средние прогулы в зависимости от внеучебной активности',
    labels={'Has_Extracurricular': 'Есть внеучебная активность',
            'Absences': 'Среднее число прогулов'},
)
fig5.update_traces(textposition='outside', marker_color='#7BC57C')
fig5.update_layout(height=420, yaxis=dict(rangemode='tozero'))
fig5.show()

## 🧮 Этап 5. Индекс Риска Отчисления (`Drop_Risk_Index`)

Уникальная формула студента (`N` — порядковый номер в группе):

$$
A = (N \bmod 5) + 1, \quad B = (N \bmod 3) + 1, \quad C = (N \bmod 4) + 1
$$

$$
\text{Drop\_Risk\_Index} = (\text{Absences} \cdot A) - (\text{StudyTimeWeekly} \cdot B) + ((4.0 - \text{GPA}) \cdot C \cdot 10)
$$

В DataLens формула добавляется внутрь VIEW. Делаем то же самое — `CREATE OR REPLACE VIEW` поверх существующего `v_student_performance_dash` дописывает новую колонку `Drop_Risk_Index` без необходимости что-то ронять или пересоздавать.

In [ ]:
# Поменяйте значение N на ваш порядковый номер в списке группы (1..200)
N = 7

A = (N % 5) + 1
B = (N % 3) + 1
C = (N % 4) + 1
print(f'N = {N}  →  A = {A},  B = {B},  C = {C}')

In [ ]:
SQL_VIEW_WITH_DRI = f'''
CREATE OR REPLACE VIEW v_student_performance_dash AS
SELECT
    StudentID, Age, StudyTimeWeekly, Absences, GPA,

    multiIf(Gender = 0, 'Мужской',
            Gender = 1, 'Женский',
            'Неизвестно') AS Gender_Name,

    multiIf(Ethnicity = 0, 'Европеоидная',
            Ethnicity = 1, 'Афроамериканская',
            Ethnicity = 2, 'Азиатская',
            'Другая') AS Ethnicity_Name,

    multiIf(ParentalEducation = 0, 'Нет',
            ParentalEducation = 1, 'Средняя школа',
            ParentalEducation = 2, 'Неоконченное высшее',
            ParentalEducation = 3, 'Бакалавр',
            ParentalEducation = 4, 'Магистр и выше',
            'Неизвестно') AS ParentalEdu_Name,

    multiIf(ParentalSupport = 0, 'Отсутствует',
            ParentalSupport = 1, 'Низкая',
            ParentalSupport = 2, 'Средняя',
            ParentalSupport = 3, 'Высокая',
            ParentalSupport = 4, 'Очень высокая',
            'Неизвестно') AS ParentalSupport_Name,

    multiIf(GradeClass = 0, 'A (Отлично)',
            GradeClass = 1, 'B (Хорошо)',
            GradeClass = 2, 'C (Удовлетворительно)',
            GradeClass = 3, 'D (Посредственно)',
            GradeClass = 4, 'F (Неудовлетворительно)',
            'Unknown') AS Grade_Name,

    Tutoring, Extracurricular, Sports, Music, Volunteering,

    if(Tutoring = 1,        'Да', 'Нет') AS Has_Tutoring,
    if(Extracurricular = 1, 'Да', 'Нет') AS Has_Extracurricular,

    -- Уникальный показатель студента: A={A}, B={B}, C={C}
    (Absences * {A}) - (StudyTimeWeekly * {B}) + ((4.0 - GPA) * {C} * 10) AS Drop_Risk_Index

FROM raw_student_performance
'''

client.command(SQL_VIEW_WITH_DRI)
print(f'✔ VIEW обновлён, добавлена колонка Drop_Risk_Index (A={A}, B={B}, C={C})')

# Проверяем DRI прямо из ClickHouse
client.query_df('''
    SELECT round(min(Drop_Risk_Index), 2)  AS dri_min,
           round(avg(Drop_Risk_Index), 2)  AS dri_avg,
           round(max(Drop_Risk_Index), 2)  AS dri_max
    FROM v_student_performance_dash
''')

### Чарт 6 (бонус). Treemap по DRI

| Параметр DataLens | Значение |
|-------------------|----------|
| Тип чарта | Treemap |
| Измерения | `ParentalEdu_Name`, `Ethnicity_Name` |
| Показатель | `AVG(Drop_Risk_Index)` |
| Цвет | `AVG(Drop_Risk_Index)`, градиент зелёный → красный |

In [ ]:
# Подтягиваем агрегат прямо из ClickHouse — это та же группировка, которую сделает DataLens
g6 = client.query_df('''
    SELECT ParentalEdu_Name,
           Ethnicity_Name,
           avg(Drop_Risk_Index)        AS DRI,
           count()                     AS students
    FROM v_student_performance_dash
    GROUP BY ParentalEdu_Name, Ethnicity_Name
    ORDER BY DRI DESC
''')
g6

In [ ]:
fig6 = px.treemap(
    g6,
    path=[px.Constant('Все студенты'), 'ParentalEdu_Name', 'Ethnicity_Name'],
    values='students',
    color='DRI',
    color_continuous_scale='RdYlGn_r',  # зелёный (низкий риск) → красный (высокий)
    color_continuous_midpoint=g6['DRI'].mean(),
    title=f'Чарт 6. Средний Drop_Risk_Index (A={A}, B={B}, C={C})',
    custom_data=['DRI', 'students'],
)
fig6.update_traces(
    texttemplate='<b>%{label}</b><br>DRI: %{customdata[0]:.1f}<br>Студентов: %{customdata[1]}',
    hovertemplate='<b>%{label}</b><br>Средний DRI: %{customdata[0]:.2f}<br>Студентов: %{customdata[1]}<extra></extra>',
)
fig6.update_layout(height=560, margin=dict(t=60, l=10, r=10, b=10))
fig6.show()

## ✅ Чек-лист перед сборкой дашборда в DataLens

- [ ] CSV прочитался и `df.shape == (2392, 15)`.
- [ ] Подключение к ClickHouse рабочее (получили версию сервера).
- [ ] `count()` в `raw_student_performance` равен `len(df)`.
- [ ] Все агрегаты сошлись (pandas vs CH).
- [ ] VIEW `v_student_performance_dash` создан, расшифрованные колонки на месте.
- [ ] Числа KPI в Чарте 1 совпадают с тем, что покажет DataLens.
- [ ] В Чарте 2 столбцы отсортированы по убыванию `Средний GPA`.
- [ ] В Чарте 3 точки **не агрегированы** (одна точка = один студент).
- [ ] В Чарте 4 включены подписи в процентах.
- [ ] В Чарте 5 видна разница между «Да» и «Нет».
- [ ] Селекторы `Gender_Name` и `Has_Tutoring` действительно меняют все 5 чартов.
- [ ] (бонус) В Treemap уникальные `A`, `B`, `C` подставлены, и градиент идёт от зелёного к красному.

После того как все галочки расставлены — чарты переносятся в DataLens по той же логике, и сборка дашборда сводится к перетаскиванию полей в нужные «полки».